In [1]:
# ============================================================
# LOAN APPROVAL & RISK ANALYTICS
# File: 04_feature_engineering.ipynb
# Purpose: Prepare ML-ready features
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("Libraries imported successfully!")


# ============================================================
# 2. LOAD CLEANED DATA
# ============================================================

DATA_PATH = "../data/loan_cleaned.csv"

df = pd.read_csv(DATA_PATH)

print("Cleaned dataset loaded successfully!")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")


# ============================================================
# 3. CREATE COPY
# ============================================================

data = df.copy()


# ============================================================
# 4. CREATE TOTAL INCOME
# ============================================================

data["TotalIncome"] = (
    data["ApplicantIncome"] +
    data["CoapplicantIncome"]
)

print("\nTotalIncome feature created.")


# ============================================================
# 5. CREATE LOG TRANSFORMED INCOME
# ============================================================

data["TotalIncomeLog"] = np.log1p(
    data["TotalIncome"]
)

print("TotalIncomeLog feature created.")


# ============================================================
# 6. CREATE LOG TRANSFORMED LOAN AMOUNT
# ============================================================

data["LoanAmountLog"] = np.log1p(
    data["LoanAmount"]
)

print("LoanAmountLog feature created.")


# ============================================================
# 7. CREATE LOAN-TO-INCOME RATIO
# ============================================================

data["LoanIncomeRatio"] = (
    data["LoanAmount"] /
    data["TotalIncome"]
)

print("LoanIncomeRatio feature created.")


# ============================================================
# 8. CREATE TOTAL INCOME PER DEPENDENT
# ============================================================

# Convert Dependents into numeric representation

data["DependentsNumeric"] = (
    data["Dependents"]
    .replace("3+", 3)
    .astype(float)
)

data["IncomePerDependent"] = (
    data["TotalIncome"] /
    (data["DependentsNumeric"] + 1)
)

print("IncomePerDependent feature created.")


# ============================================================
# 9. CREATE LOAN BURDEN CATEGORY
# ============================================================

def loan_burden_category(ratio):

    if ratio < 0.20:
        return "Low"

    elif ratio < 0.40:
        return "Medium"

    else:
        return "High"


data["LoanBurden"] = (
    data["LoanIncomeRatio"]
    .apply(loan_burden_category)
)

print("LoanBurden feature created.")


# ============================================================
# 10. ENCODE TARGET VARIABLE
# ============================================================

data["Loan_Status"] = (
    data["Loan_Status"]
    .map({
        "Y": 1,
        "N": 0
    })
)

print("\nLoan_Status encoded:")
print(data["Loan_Status"].value_counts())


# ============================================================
# 11. CHECK CATEGORICAL COLUMNS
# ============================================================

categorical_columns = data.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nCategorical columns:")
print(categorical_columns)


# ============================================================
# 12. ONE-HOT ENCODING
# ============================================================

data_encoded = pd.get_dummies(
    data,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

print("\nOne-hot encoding completed.")


# ============================================================
# 13. DISPLAY FEATURES
# ============================================================

print("\n" + "=" * 60)
print("FEATURES AFTER ENCODING")
print("=" * 60)

print(data_encoded.columns.tolist())


# ============================================================
# 14. CHECK DATASET SHAPE
# ============================================================

print("\n" + "=" * 60)
print("DATASET SHAPE AFTER FEATURE ENGINEERING")
print("=" * 60)

print(
    f"Rows    : {data_encoded.shape[0]}"
)

print(
    f"Columns : {data_encoded.shape[1]}"
)


# ============================================================
# 15. CHECK MISSING VALUES
# ============================================================

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

print(
    data_encoded.isnull().sum()[
        data_encoded.isnull().sum() > 0
    ]
)

print(
    "\nTotal missing values:",
    data_encoded.isnull().sum().sum()
)


# ============================================================
# 16. SEPARATE FEATURES AND TARGET
# ============================================================

X = data_encoded.drop(
    columns=["Loan_Status"]
)

y = data_encoded["Loan_Status"]


print("\n" + "=" * 60)
print("FEATURES AND TARGET")
print("=" * 60)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")


# ============================================================
# 17. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n" + "=" * 60)
print("TRAIN TEST SPLIT")
print("=" * 60)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")


# ============================================================
# 18. FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

print("\nFeature scaling completed.")


# ============================================================
# 19. CONVERT BACK TO DATAFRAME
# ============================================================

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)


# ============================================================
# 20. DISPLAY TRAINING DATA
# ============================================================

print("\n" + "=" * 60)
print("SCALED TRAINING DATA")
print("=" * 60)

print(X_train_scaled.head())


# ============================================================
# 21. DISPLAY TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)
print("TARGET DISTRIBUTION")
print("=" * 60)

print(
    y_train.value_counts()
)

print("\nTarget Percentage:")

print(
    y_train
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


# ============================================================
# 22. SAVE ML DATASET
# ============================================================

ML_DATA_PATH = "../data/loan_ml_ready.csv"

data_encoded.to_csv(
    ML_DATA_PATH,
    index=False
)

print(
    f"\nML-ready dataset saved at: {ML_DATA_PATH}"
)


# ============================================================
# 23. SAVE TRAINING DATA
# ============================================================

train_data = X_train_scaled.copy()

train_data["Loan_Status"] = y_train

train_data.to_csv(
    "../data/train_data.csv",
    index=False
)

print("Training dataset saved.")


# ============================================================
# 24. SAVE TESTING DATA
# ============================================================

test_data = X_test_scaled.copy()

test_data["Loan_Status"] = y_test

test_data.to_csv(
    "../data/test_data.csv",
    index=False
)

print("Testing dataset saved.")


# ============================================================
# 25. FINAL FEATURE SUMMARY
# ============================================================

feature_summary = pd.DataFrame({
    "Feature": X.columns,
    "Data_Type": X.dtypes.astype(str).values
})

print("\n" + "=" * 60)
print("FINAL FEATURE SUMMARY")
print("=" * 60)

print(feature_summary)


# ============================================================
# 26. SAVE FEATURE SUMMARY
# ============================================================

feature_summary.to_csv(
    "../data/feature_summary.csv",
    index=False
)

print("\nFeature summary saved successfully!")


# ============================================================
# 27. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("FEATURE ENGINEERING VALIDATION")
print("=" * 60)

print(f"Original features : {df.shape[1] - 1}")
print(f"ML features       : {X.shape[1]}")
print(f"Training rows     : {X_train.shape[0]}")
print(f"Testing rows      : {X_test.shape[0]}")
print(
    f"Missing values    : "
    f"{data_encoded.isnull().sum().sum()}"
)

print("\n✓ Feature engineering completed successfully!")
print("✓ Dataset is ready for model training.")

Libraries imported successfully!
Cleaned dataset loaded successfully!
Rows    : 614
Columns : 12

TotalIncome feature created.
TotalIncomeLog feature created.
LoanAmountLog feature created.
LoanIncomeRatio feature created.
IncomePerDependent feature created.
LoanBurden feature created.

Loan_Status encoded:
Loan_Status
1    422
0    192
Name: count, dtype: int64

Categorical columns:
['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'LoanBurden']

One-hot encoding completed.

FEATURES AFTER ENCODING
['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Loan_Status', 'TotalIncome', 'TotalIncomeLog', 'LoanAmountLog', 'LoanIncomeRatio', 'DependentsNumeric', 'IncomePerDependent', 'Gender_Male', 'Married_Yes', 'Dependents_1', 'Dependents_2', 'Dependents_3+', 'Education_Not Graduate', 'Self_Employed_Yes', 'Property_Area_Semiurban', 'Property_Area_Urban']

DATASET SHAPE AFTER FEATURE ENGINEERING
Rows    : 614
Columns : 2

C:\Users\Princ\AppData\Local\Temp\ipykernel_16348\180772.py:151: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = data.select_dtypes(
